In [31]:
%reset -f
import arcpy
arcpy.ResetEnvironments()
import os
from arcpy.sa import ExtractByMask

# check out Spatial Analyst license
arcpy.CheckOutExtension("Spatial")

'CheckedOut'

In [32]:
# ── PATHS ────────────────────────────────────────────────────────────────────────────────────────────

# Input boundary polygon with extent to clip raster to
boundary = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Thesis_geodatabases\MASTER_THESIS.gdb\boundary_thesis_p26920"

# Input raster "GBC15a_bathy_entireEastCarib_g4326"
unclip_gbc_bathy = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec.gdb\GBC15a_bathy_entireEastCarib_g4326"

# Output reprojected and reprojected/clipped rasters
reproj_unclip_gbc_bathy = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec.gdb\GBC15a_bathy_entireEastCarib_p26920_v6642"
gbc_bathy = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec_thesisextent.gdb\GBC15a_bathy_p26920_v6642"

In [33]:
# ── SETTINGS ─────────────────────────────────────────────────────────────────────────────────────────
arcpy.env.overwriteOutput = True    # overwrite existing outputs on rerun
arcpy.env.compression = "NONE"      # no compression on output rasters

In [34]:
# ── INSPECT RASTER DATA BEFORE REPROJECTING ──────────────────────────────────────────────────────────

# Function to print all info (number of bands, pixel type, min, max, cell size, coordinate systems) for each raster
def print_raster_info(raster):

    # get number of bands info
    d = arcpy.Describe(raster)
    # get pixel type/min/max/cell size info (r = arcpy.Raster() loads raster dataset into a Raster object so arcpy can access stats)
    r = arcpy.Raster(raster)
    # get spatial reference info
    sr = d.spatialReference
    vcs = sr.VCS.name if sr.VCS else "NONE"

    # info print block for raster
    print(f"\n---------------- {os.path.basename(raster)} ----------------: \n")
    print(f"   Number bands = {d.bandCount}")
    print(f"   Pixel type = {r.pixelType}")
    print(f"   ⬇️ Min = {r.minimum}")
    print(f"   ⬆️ Max = {r.maximum}")
    print(f"   📐 Cell size = {r.meanCellWidth}")
    print(f"   Horizontal SR: {sr.name}")
    print(f"   Vertical SR: {vcs}")
    print(f"\n")


# Calculate Statistics (Data Management Tools): (default X/Y skip factor = 1; skip raster if stats already calculated)
arcpy.management.CalculateStatistics(
    in_raster_dataset=unclip_gbc_bathy,
    skip_existing="SKIP_EXISTING"
)

# Call function to print info for input raster
print_raster_info(unclip_gbc_bathy)


---------------- GBC15a_bathy_entireEastCarib_g4326 ----------------: 

   Number bands = 1
   Pixel type = S16
   ⬇️ Min = -8733.0
   ⬆️ Max = 0.0
   📐 Cell size = 0.0041666666666666605
   Horizontal SR: GCS_WGS_1984
   Vertical SR: NONE




In [35]:
# ── 1. REPROJECT FULL GBC RASTER TO EPSG:26920 / EPSG:6642 ───────────────────────────────────────────

# Project Raster (Data Management Tools):
try:
    arcpy.management.ProjectRaster(
        in_raster=unclip_gbc_bathy,
        out_raster=reproj_unclip_gbc_bathy,
        out_coor_system=arcpy.SpatialReference(26920, 6642),
        resampling_type="BILINEAR",
        geographic_transform="WGS_1984_(ITRF00)_To_NAD_1983",
        # leave NoData as default ("") to preserve NoData (land) from input raster
    )
    print(f"✅ Reprojected raster created: {reproj_unclip_gbc_bathy} \n")

# print warning message if tool fails
except Exception as e:
    print(f"\n❌[FAILURE] Reprojection failed: {e} \n")

✅ Reprojected raster created: C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec.gdb\GBC15a_bathy_entireEastCarib_p26920_v6642 



In [36]:
# ── INSPECT RASTER DATA AFTER REPROJECTING ───────────────────────────────────────────────────────────

# Calculate Statistics (Data Management Tools): (default X/Y skip factor = 1)
arcpy.management.CalculateStatistics(in_raster_dataset=reproj_unclip_gbc_bathy)

# Call function to print info for reprojected raster
print_raster_info(reproj_unclip_gbc_bathy)


---------------- GBC15a_bathy_entireEastCarib_p26920_v6642 ----------------: 

   Number bands = 1
   Pixel type = S16
   ⬇️ Min = -8619.0
   ⬆️ Max = 0.0
   📐 Cell size = 472.8452741843477
   Horizontal SR: NAD_1983_UTM_Zone_20N
   Vertical SR: VIVD09_height




In [37]:
# ── 2. CLIP REPROJECTED RASTER TO THESIS BOUNDARY POLYGON ────────────────────────────────────────────

# set environment coordinate system: horizontal EPSG:26920 + vertical EPSG:6642 (Extract by Mask tool does NOT carry vertical coordinate system of input(s) through to output layer(s), even if the vert coord system is present in all input layers - so must manually set coordinate system of output(s) to have both projected/geographic + vertical coordinate systems)
arcpy.env.outputCoordinateSystem = arcpy.SpatialReference(26920, 6642)
# snap raster to input raster grid so output aligns to same grid
arcpy.env.snapRaster = reproj_unclip_gbc_bathy


# Extract by Mask (Spatial Analyst Tools): 
try:
    outExtractByMask = ExtractByMask(
        in_raster=reproj_unclip_gbc_bathy,
        # clip input raster to extent of boundary_thesis_p26920 polygon
        in_mask_data=boundary,
        # keep raster cells that are inside (aka overlap with) boundary polygon
        extraction_area="INSIDE"
    )
    # save output as new raster
    outExtractByMask.save(gbc_bathy)
    print(f"✅ Clipped bathy raster exported: {gbc_bathy} \n")

# print warning message if tool fails
except Exception as e:
    print(f"\n❌[FAILURE] Clipping raster to thesis boundary failed: {e}")


# Clear export settings in environment for subsequent blocks
arcpy.ClearEnvironment("outputCoordinateSystem")
arcpy.ClearEnvironment("snapRaster")

✅ Clipped bathy raster exported: C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec_thesisextent.gdb\GBC15a_bathy_p26920_v6642 



In [38]:
# ── INSPECT RASTER DATA AFTER CLIPPING EXTENT TO THESIS BOUNDARY ─────────────────────────────────────

# Calculate Statistics (Data Management Tools): (default X/Y skip factor = 1)
arcpy.management.CalculateStatistics(in_raster_dataset=gbc_bathy)

# Call function to print info for clipped raster
print_raster_info(gbc_bathy)


---------------- GBC15a_bathy_p26920_v6642 ----------------: 

   Number bands = 1
   Pixel type = S16
   ⬇️ Min = -8619.0
   ⬆️ Max = 0.0
   📐 Cell size = 472.8452741843477
   Horizontal SR: NAD_1983_UTM_Zone_20N
   Vertical SR: VIVD09_height


